# Gemini RAG Knowledge Assistant using LangChain
**UST Assignment 6 | Knowledge base: Contoso Platform Troubleshooting Runbook**

This notebook uses **LangChain** to connect the RAG components:
- PDF loading: PyMuPDF via `PyMuPDFLoader`
- Text splitting: `RecursiveCharacterTextSplitter`
- Document and query embeddings: Gemini `gemini-embedding-001`
- Vector store: FAISS
- Answer generation: Gemini 3 Flash (`gemini-3-flash-preview`)

Run cells from top to bottom. The supplied runbook is in `data/troubleshooting_runbook.pdf`. Your API key is requested privately and is not saved in this notebook.

**Testing is intentionally limited:** three simple questions and a Top-K retrieval comparison. Document embeddings are generated once; avoid rerunning that cell unless the PDF changes.

## 1. Install required packages

In [ ]:
%pip -q install -U langchain langchain-community langchain-google-genai langchain-text-splitters langchain-classic pymupdf faiss-cpu

## 2. Imports and Gemini configuration

In [ ]:
import os, getpass, time
from pathlib import Path

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS

print("LangChain imports completed.")

In [ ]:
api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
if not api_key:
    api_key = getpass.getpass("Enter Gemini API key (hidden): ")

os.environ["GOOGLE_API_KEY"] = api_key

EMBEDDING_MODEL = "models/gemini-embedding-001"
GENERATION_MODEL = "gemini-3-flash-preview"

embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL,
    task_type="RETRIEVAL_DOCUMENT"
)
llm = ChatGoogleGenerativeAI(
    model=GENERATION_MODEL,
    temperature=0.2,
    max_output_tokens=300,
    timeout=60,
    max_retries=2
)

print("Gemini embedding and generation models configured.")

## 3. Load the troubleshooting PDF

In [ ]:
pdf_path = Path("data/troubleshooting_runbook.pdf")

if not pdf_path.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        pdfs = [name for name in uploaded if name.lower().endswith(".pdf")]
        if not pdfs:
            raise FileNotFoundError("Upload the troubleshooting runbook PDF.")
        pdf_path = Path(pdfs[0])
    except ImportError:
        pdf_path = Path(input("Enter PDF file path: ").strip()).expanduser()

if not pdf_path.exists():
    raise FileNotFoundError(f"PDF not found: {pdf_path}")

loader = PyMuPDFLoader(str(pdf_path))
documents = loader.load()

print("Pages loaded:", len(documents))
print("Total extracted characters:", sum(len(doc.page_content) for doc in documents))
print("First page preview:\\n", documents[0].page_content[:1000])


## 4. Split the document into chunks

In [ ]:
# Approximate character-based chunk size; not an exact token count.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=2800,
    chunk_overlap=350
)
chunks = splitter.split_documents(documents)

print("Chunks created:", len(chunks))
print("Example chunk metadata:", chunks[0].metadata)
print("Example chunk text:\\n", chunks[0].page_content[:800])

## 5. Create Gemini embeddings and FAISS vector store

In [ ]:
# This is the main document-embedding API step. Run it once per document version.
# FAISS stores vectors locally after embedding.
vector_store = FAISS.from_documents(chunks, embeddings)

print("FAISS vector store created.")
print("Number of indexed chunks:", len(chunks))

## 6. Retrieve relevant chunks

In [ ]:
def retrieve(question, top_k=3):
    return vector_store.similarity_search_with_score(question, k=top_k)

# Retrieval preview
question = "What causes DB-POOL errors and how can I fix them?"
results = retrieve(question, top_k=3)

for rank, (doc, score) in enumerate(results, start=1):
    page = doc.metadata.get("page", "unknown")
    print(f"\\n{rank}. Page {page} | distance={score:.4f}")
    print(doc.page_content[:650])

## 7. Generate an answer with Gemini 3 Flash

In [ ]:
def ask_rag(question, top_k=2):
    results = retrieve(question, top_k=top_k)

    context = "\\n\\n".join(
        f"[Source: page {doc.metadata.get('page', 0) + 1}]\\n{doc.page_content[:1200]}"
        for doc, score in results
    )

    prompt = f"""
You are a troubleshooting assistant. Answer using ONLY the retrieved runbook context.
Do not invent facts, commands, causes, or fixes.
If the context is insufficient, say so.
Keep the answer short (3-5 sentences) and cite source page(s).

CONTEXT:
{context}

QUESTION:
{question}
"""

    response = llm.invoke(prompt)
    answer = response.content
    return answer, results

question = "What causes DB-POOL errors and how can I fix them?"
answer, evidence = ask_rag(question, top_k=2)

print("QUESTION:", question)
print("\\nANSWER:\\n", answer)

## 8. Limited end-to-end testing

In [ ]:
# Three simple tests to demonstrate multiple queries.
test_questions = [
    "How do I troubleshoot an expired authentication token?",
    "What should I check when a service returns HTTP 500?",
    "How can I resolve a database connection pool error?"
]

for question in test_questions:
    answer, evidence = ask_rag(question, top_k=2)
    source_pages = sorted({
        doc.metadata.get("page", "unknown") for doc, score in evidence
    })
    print("\\nQUESTION:", question)
    print("ANSWER:", answer)
    print("SOURCE PAGES:", source_pages)
    print("-" * 60)

## 9. Top-K retrieval comparison

In [ ]:
# Retrieval-only comparison. This does not call Gemini for answer generation.
question = "How do I troubleshoot an expired authentication token?"

for k in [3, 5, 7]:
    results = retrieve(question, top_k=min(k, len(chunks)))
    pages = [doc.metadata.get("page", "unknown") for doc, score in results]
    print(f"Top-K={k}: retrieved pages {pages}")

## 10. Assignment summary

1. **Preprocessing:** loaded the runbook PDF, extracted page text and split it into overlapping chunks using LangChain.
2. **Embeddings and vector database:** generated document/query embeddings using Gemini `gemini-embedding-001` and stored vectors in FAISS.
3. **Retrieval:** used LangChain similarity search and compared Top-K values 3, 5 and 7.
4. **Answer generation:** used Gemini 3 Flash to answer from retrieved context and cite source pages.
5. **End-to-end testing:** tested three simple troubleshooting questions.

### Fill in after running
- Number of pages loaded: **[actual output]**
- Number of chunks: **[actual output]**
- Top-K that worked best in your inspection: **[3 / 5 / 7]**
- Did answers match the runbook? **[brief observation]**

**Cost control:** document embeddings are generated once. Three test questions make three Gemini generation requests. The Top-K comparison performs retrieval only. Avoid rerunning the FAISS creation cell unless the PDF or chunking changes.

### References
- UST Assignment 6: Retrieval-Augmented Generation (RAG).
- Contoso Platform Troubleshooting Runbook, version 3.2.
- [LangChain documentation](https://python.langchain.com/)
- [Gemini API documentation](https://ai.google.dev/gemini-api/docs)